In [18]:
import pandas as pd
from collections import Counter

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)

This notebook uses the AllCalls data to analyze redirections at the ends of calls to identify potential patterns. 

This notebook specifically focuses on redirection patterns at the ends of calls that reach the 8300 number at the end (i.e., `Called number == '13123478300'` in the last leg). However, the code in here could be used to see redirection patterns for any ending called number.

This notebook outputs two tables containing: 
- summary information about just the redirecting numbers in the last legs of calls that end with the 8300 number
- summary information about the redirecting numbers and redirect reasons in the last legs of calls that end with the 8300 number

This will help simplify the dashboard-making process.

# Manual inputs

### Importing files
One file needed:
- the combined csv of all AllCalls data

In the following code, manually input the path to the file on your computer.

In [19]:
# Read AllCalls dataset (with all cols as str dtype)
allcalls = pd.read_csv('Data/AllCallsData_combined.csv', dtype=str, na_filter=True)

### Choose ending number
You can adjust `ending_number` to be any called number to focus on as the last called number in calls. Variable names and comments in the notebook assume that it is the 8300 number, but the code itself should work for any ending number.

In [20]:
# Looking at calls that end with the following ending number as the called number in the call's last leg
ending_number = '13123478300'

# Cleaning and processing

In [21]:
# Get rid of columns that are entirely NaN values
allcalls_no_nan = allcalls.dropna(axis=1, how='all')

# Update dtypes for columns that should not be represented as strings
allcalls_no_mixed = allcalls_no_nan.copy()

numeric_cols = ['Ring duration', 'Duration']
datetime_cols = ['Start time', 'Release time', 'Answer time', 'Report time']

for col in numeric_cols:
    if col in allcalls_no_mixed.columns:
        allcalls_no_mixed[col] = allcalls_no_mixed[col].astype(float)

for col in datetime_cols:
    if col in allcalls_no_mixed.columns:
        allcalls_no_mixed[col] = pd.to_datetime(allcalls_no_mixed[col], format='mixed')

**NOTE:** Although `dtype=str` is specified in the `read_csv` function to convert all the data into strings, `NaN` values are still left as floats. This means that some columns still technically have mixed data types.

In [22]:
# Sort data by correlation ID & start time
allcalls_sorted = allcalls_no_mixed.sort_values(by=['Correlation ID', 'Start time'])

The sorting means that within each correlation ID, rows are now ordered chronologically with the first (uppermost) row being the first leg (start) of the call.

**NOTE:** If two rows have the same correlation ID and start time, the original order that they appeared in before sorting is preserved. 

In [23]:
# Select only cols relevant for visualizing the distribution of original/redirect reason for different called numbers
allcalls_reasons = allcalls_sorted[['Correlation ID', 'Called number', 'Redirecting number', 
                                    'Original reason', 'Related reason', 'Redirect reason', 
                                    'Inbound trunk', 'Outbound trunk', 'Direction', 
                                    'Call type', 'Client type', 'User type', 'Start time']].copy()

# Analysis

In [24]:
# Get the sequences of called number AND redirecting number together for ALL calls
allcalls_crr = allcalls_reasons.loc[:, ['Correlation ID', 'Called number', 'Redirecting number', 'Redirect reason']].copy()
allcalls_crr['combined'] = allcalls_crr[['Called number', 'Redirecting number', 'Redirect reason']].apply(tuple, axis=1)   # Combine multiple cols into a col of tuples

sequences_all_crr = []

for group in allcalls_crr[['Correlation ID', 'combined']].groupby('Correlation ID'):
    id, df = group[0], group[1]
    sequences_all_crr.append(tuple(df['combined']))

# Show example of how a call's journey between different called numbers is represented
display(sequences_all_crr[:5])

[(('13123411070', nan, nan),),
 (('13124312299', nan, nan),),
 (('13123478311', nan, nan),
  ('13123478300', '13123478311', 'NoAnswer'),
  ('13123478300', '13123411070', 'NoAnswer')),
 (('13122296344', nan, nan),
  ('13123478300', '13122296344', 'Unconditional'),
  ('13123478300', '13122296344', 'Unconditional')),
 (('17086568223', nan, nan),)]

### Look at calls that end with provided ending number

In [25]:
# Get sequences of called & redirecting number that end with 8300 as the called number in last leg
sequences_all_crr_end8300 = [seq for seq in sequences_all_crr if seq[-1][0] == ending_number]

# Get the redirecting numbers corresponding to the ending legs with 8300
sequences_all_crr_end8300_rnum = [seq[-1][1] for seq in sequences_all_crr_end8300]

# Calculate the proportion of all calls that end with 8300
print(f'Number of calls ending in {ending_number[-4:]}:\t\t{len(sequences_all_crr_end8300)}')
print(f'Number of all unique correlation IDs:\t{len(allcalls_reasons['Correlation ID'].unique())}')
print(f'Proportion of calls that end with {ending_number[-4:]}:\t{len(sequences_all_crr_end8300)/len(allcalls_reasons['Correlation ID'].unique())}')

Number of calls ending in 8300:		117579
Number of all unique correlation IDs:	516786
Proportion of calls that end with 8300:	0.22751970835123242


## Create tables for dashboard

### Get the more complex table with both redirecting number & redirect reason

In [ ]:
# Get Counter of tuple (Redirecting number, Redirect reason) from the last legs of each call
table = Counter([seq[-1][-2:] for seq in sequences_all_crr_end8300])

# Convert Counter object to df
df = pd.DataFrame(list(table.items()), columns=['combined', 'Count'])

# Split col of tuples into separate cols
df[['Redirecting number', 'Redirect reason']] = df['combined'].apply(pd.Series)

# Remove col of tuples
df.drop(columns=['combined'], inplace=True)

# Reorder cols
df = df[['Redirecting number', 'Redirect reason', 'Count']]

# Convert NaNs to string in Redirecting number & Redirect reason cols
df = df.astype({'Redirecting number': str, 'Redirect reason': str})

# Sort by redirecting number according to their total count, then sort by count for each redirect reason
counter_all_crr_end8300_rnum = Counter(sequences_all_crr_end8300_rnum)
counter_all_crr_end8300_rnum = Counter({str(k):v for k,v in counter_all_crr_end8300_rnum.items()})  # Convert nan to str
order_by_totals = list(counter_all_crr_end8300_rnum.keys())

df['Redirecting number'] = pd.Categorical(df['Redirecting number'], categories=order_by_totals, ordered=True)
df = df.sort_values(['Redirecting number', 'Count'], ascending=[True, False])
df.reset_index(drop=True, inplace=True)

# Create col of proportion of all 8300-ending calls from the Count col
number_of_end8300_calls = len(sequences_all_crr_end8300)
df[f'Proportion of {ending_number[-4:]}-ending calls'] = df['Count'] / number_of_end8300_calls

# Create col with only the last 4 digits of the redirecting number
df['Redirecting number (truncated)'] = df['Redirecting number'].apply(lambda x: x[-4:])

# Reorder cols again
df = df[['Redirecting number', 'Redirecting number (truncated)', 'Redirect reason', 'Count', f'Proportion of {ending_number[-4:]}-ending calls']]

# Add col for percentage labels for each redirecting number + redirect reason combination
df[f'Percent of {ending_number[-4:]}-ending calls'] = df[f'Proportion of {ending_number[-4:]}-ending calls'].apply(lambda x: f'{x*100:.1f}%')

# Display results
df.head(10)

,Redirecting number,Redirecting number (truncated),Redirect reason,Count,Proportion of 8300-ending calls,Percent of 8300-ending calls
0,13123411070,1070,NoAnswer,23306,0.198216,19.8%
1,13123411070,1070,UserBusy,5273,0.044846,4.5%
2,13123411070,1070,Unavailable,3766,0.032030,3.2%
3,13123411070,1070,Unconditional,49,0.000417,0.0%
4,13122296344,6344,Unconditional,6623,0.056328,5.6%
5,13123478302,8302,NoAnswer,210,0.001786,0.2%
6,13123478302,8302,Unavailable,49,0.000417,0.0%
7,13123478302,8302,UserBusy,15,0.000128,0.0%
8,13122296306,6306,NoAnswer,273,0.002322,0.2%
9,nan,nan,nan,5104,0.043409,4.3%


### Get the simpler table with only redirecting number

In [43]:
# Copy complex table
df_simple = df.copy()

# Aggregate the data
df_simple = df_simple.groupby('Redirecting number', as_index=False, observed=False)['Count'].sum()

# Add the cols that were removed
df_simple[f'Proportion of {ending_number[-4:]}-ending calls'] = df_simple['Count'] / number_of_end8300_calls

# Create col with only the last 4 digits of the redirecting number
df_simple['Redirecting number (truncated)'] = df_simple['Redirecting number'].apply(lambda x: x[-4:])

# Add col for percentage labels for each redirecting number
df_simple[f'Percent of {ending_number[-4:]}-ending calls'] = df_simple[f'Proportion of {ending_number[-4:]}-ending calls'].apply(lambda x: f'{x*100:.1f}%')

# Reorder cols
df_simple = df_simple[['Redirecting number', 'Redirecting number (truncated)', 'Count', f'Proportion of {ending_number[-4:]}-ending calls', f'Percent of {ending_number[-4:]}-ending calls']]

df_simple.head(10)

,Redirecting number,Redirecting number (truncated),Count,Proportion of 8300-ending calls,Percent of 8300-ending calls
0,13123411070,1070,32394,0.275508,27.6%
1,13122296344,6344,6623,0.056328,5.6%
2,13123478302,8302,274,0.002330,0.2%
3,13122296306,6306,273,0.002322,0.2%
4,nan,nan,5104,0.043409,4.3%
5,13124235904,5904,1328,0.011295,1.1%
6,13122296071,6071,4085,0.034743,3.5%
7,13122296078,6078,380,0.003232,0.3%
8,13122296079,6079,6266,0.053292,5.3%
9,13124235938,5938,27136,0.230790,23.1%


# Exporting

You can adjust where the file is exported to.

In [44]:
df.to_csv('Data/AllCallsData_8300EndRedirects.csv', index=False)
df_simple.to_csv('Data/AllCallsData_8300EndRedirectsAggregate.csv', index=False)